# GRIP-Benchmark-34 — Multi-Model Evaluation

Runs the 10 open-source vision-language models discussed earlier against GRIP's 34 domains
(500,000 closed L1–L5 questions, plus an optional pass over the 100,000 open-ended questions).

**Before running this notebook:**

1. **Pull the real images.** This checkout stores PNGs via Git LFS; until you run
   `git lfs install && git lfs pull` from a terminal, `images/*.png` files are small text
   pointers, not real image bytes. Step 1 below detects this and will refuse to query a model
   with a pointer file.
2. **Set API keys.** Create a `.env` file at the repo root (or export environment variables)
   with `OPENROUTER_API_KEY=...` for the OpenRouter-hosted models. A few models in the registry
   (Kimi-VL, DeepSeek-VL2, Molmo2, SpatialStack/G2VLM) are not reliably available on a hosted
   API at the time of writing — the registry points them at a local OpenAI-compatible endpoint
   (e.g. a `vLLM`/`sglang` server you run yourself) and ships **disabled** by default.
3. **Start small.** `SMOKE_TEST = True` runs 1 model x 1 domain x a couple of images first.
   The full suite is 500,000 questions x 10 models = 5,000,000 model calls — expect this to be
   slow and expensive. Scale up gradually via `SAMPLE_PER_LEVEL` before ever setting
   `SMOKE_TEST = False` with all models enabled.
4. **Automated scoring is approximate**, especially for free-form open-ended answers. Treat the
   accuracy numbers here as a first-pass signal and spot-check a sample of `raw_response` values
   before reporting results.

All results are cached to `eval_results/<model_key>/<domain>.jsonl` and are resumable — rerunning
a cell skips questions that already have a cached answer.

In [ ]:
%pip install -q pandas numpy pillow requests tqdm matplotlib

In [ ]:
import base64
import json
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm

pd.set_option("display.width", 140)

In [ ]:
# ---- Locate the repo root (works whether the notebook is opened from the root or a subfolder) ----
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "Dataset").exists():
    for parent in Path.cwd().parents:
        if (parent / "Dataset").exists():
            REPO_ROOT = parent
            break

DATASET_ROOT = REPO_ROOT / "Dataset"
RESULTS_ROOT = REPO_ROOT / "eval_results"
RESULTS_ROOT.mkdir(exist_ok=True)

# ---- Sampling knobs -------------------------------------------------------
# GRIP has 500,000 closed questions across 34 domains. SAMPLE_PER_LEVEL images
# are drawn per difficulty level per domain (5 levels), so a domain contributes
# SAMPLE_PER_LEVEL * 5 questions per model. Raise this gradually.
SAMPLE_PER_LEVEL = 5
RANDOM_SEED = 0
MAX_WORKERS = 4            # concurrent requests per model per domain
REQUEST_TIMEOUT_S = 90
MAX_RETRIES = 4

# ---- Smoke test ------------------------------------------------------------
# Keep this True until you've confirmed the pipeline works end-to-end on one
# cheap model and one domain. Flip to False only when you're ready to spend.
SMOKE_TEST = True
SMOKE_TEST_DOMAINS = ["angle_estimation"]
SMOKE_TEST_MODELS = ["qwen3-vl-8b-instruct"]
SMOKE_TEST_SAMPLE_PER_LEVEL = 2

print(f"Repo root:    {REPO_ROOT}")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Results root: {RESULTS_ROOT}")

In [ ]:
def load_dotenv_simple(env_path: Path) -> None:
    """Minimal .env loader so API keys don't need to be exported manually."""
    if not env_path.is_file():
        return
    for raw in env_path.read_text(encoding="utf-8-sig").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_dotenv_simple(REPO_ROOT / ".env")
print("OPENROUTER_API_KEY set:", bool(os.environ.get("OPENROUTER_API_KEY")))
print("MODELSCOPE_API_KEY set:", bool(os.environ.get("MODELSCOPE_API_KEY")),
      "(needed for the free-tier Qwen3-VL-235B, Qwen3-VL-8B, and InternVL3.5 routes)")

## Step 1 — Verify images are real PNG bytes, not Git LFS pointers

`.gitattributes` marks every `*.png` as an LFS-tracked file. If `git lfs pull` was never run,
`images/*.png` files are ~130-byte text pointers (`version https://git-lfs.github.com/spec/v1 ...`)
instead of actual images. Sending a pointer file to a vision model wastes a request and produces
a meaningless answer, so every loader in this notebook filters these out and warns loudly instead.

In [ ]:
def is_lfs_pointer(path: Path, probe_bytes: int = 64) -> bool:
    try:
        with open(path, "rb") as f:
            head = f.read(probe_bytes)
        return head.startswith(b"version https://git-lfs.github.com/spec")
    except FileNotFoundError:
        return False


def check_images_pulled(sample_domains=None, sample_per_domain: int = 3) -> dict:
    domains = sample_domains or sorted(p.name for p in DATASET_ROOT.glob("*_dataset_*") if p.is_dir())
    report = {}
    for domain in domains:
        images_dir = DATASET_ROOT / domain / "images"
        if not images_dir.exists():
            report[domain] = "no images/ folder found"
            continue
        sample_files = list(images_dir.glob("*.png"))[:sample_per_domain]
        if not sample_files:
            report[domain] = "no PNGs found"
            continue
        pointers = sum(is_lfs_pointer(f) for f in sample_files)
        report[domain] = "LFS POINTER (not pulled)" if pointers else "OK (real image bytes)"
    return report


lfs_report = check_images_pulled()
for domain, status in lfs_report.items():
    print(f"{domain:32s} {status}")

if any("POINTER" in v for v in lfs_report.values()):
    print("\n" + "=" * 78)
    print("Images are Git LFS pointers in this checkout. Run in a terminal, then re-run this cell:")
    print("    git lfs install")
    print("    git lfs pull")
    print("=" * 78)

## Step 2 — Discover all 34 domains

Domains are discovered by the presence of `build_manifest.json`, the same signal
`Dataset/rebuild_combined_suite_files.py` uses, so this stays in sync with however many
domain folders actually exist under `Dataset/`.

In [ ]:
def discover_domains() -> dict:
    domains = {}
    for manifest_path in sorted(DATASET_ROOT.glob("*_dataset_*/build_manifest.json")):
        domain_dir = manifest_path.parent
        domains[domain_dir.name] = {
            "dir": domain_dir,
            "images_dir": domain_dir / "images",
            "question_set": domain_dir / "question_set.csv",
            "answer_key": domain_dir / "answer_key.csv",
            "annotations": domain_dir / "annotations.jsonl",
            "open_questions": domain_dir / "open_questions.csv",
            "open_answer_key": domain_dir / "open_answer_key.csv",
            "manifest": manifest_path,
        }
    return domains


ALL_DOMAINS = discover_domains()
print(f"Discovered {len(ALL_DOMAINS)} domains:")
for name in ALL_DOMAINS:
    print(" -", name)

## Step 3 — Model registry (the 10 models)

Two backends:

- `"openrouter"` — OpenRouter's OpenAI-compatible API. Requires `OPENROUTER_API_KEY`.
- `"openai_compatible"` — any OpenAI-compatible server: a hosted free tier (ModelScope) or
  something you self-host (`vLLM`, `sglang`).

**Cost/access summary** (checked at time of writing — verify before a large run):

| Model | Route | Cost |
|---|---|---|
| Qwen3-VL 235B-A22B Instruct | ModelScope | **Free** (2,000 req/day shared cap, needs Alibaba Cloud real-name verification) |
| Qwen3-VL 235B-A22B Thinking | OpenRouter | $0.20 / $0.88 per 1M — not confirmed free anywhere |
| Qwen3-VL 8B Instruct | ModelScope | **Free** |
| InternVL3.5 241B-A28B | ModelScope | **Free** — not on OpenRouter at all |
| GLM-4.6V | OpenRouter | $0.30 / $0.90 per 1M — GLM chat models are free on ModelScope, 4.6V specifically unconfirmed |
| Pixtral Large | OpenRouter | $2.00 / $6.00 per 1M — no free route found anywhere |
| Llama 4 Maverick | OpenRouter `:free` slug | **Free**, rate-limited, shares OpenRouter's global free quota (ModelScope also lists it free as an alternative) |
| Kimi-VL-A3B-Thinking | self-host | **Free GPU-hours** (Colab/Kaggle) — only 16B total/3B active, not on any hosted free API |
| DeepSeek-VL2 | self-host | **Free GPU-hours** — tiny variant is 1B activated, trivial to run |
| Molmo2 | self-host | **Free GPU-hours** — Western lab, not on ModelScope |

ModelScope's free tier (`https://api-inference.modelscope.cn/v1`) requires its own
`MODELSCOPE_API_KEY` and an Alibaba Cloud account with real-name verification — a real
barrier for some users, but the trade-off for genuinely $0 access to four 100B+ parameter
models. `price_per_1m_usd` is recorded on every entry so cost stays visible at a glance.
`verified: False` means the slug wasn't directly confirmed. Run the validation cell below
before a large/expensive run. Models are `enabled: False` by default when they need a local
endpoint you haven't set up yet.

In [ ]:
# ModelScope (Alibaba's model hub) runs a genuinely free OpenAI-compatible API-Inference
# tier -- https://api-inference.modelscope.cn/v1, 2,000 requests/day total, <=500/model,
# dynamic concurrency -- and it happens to host most of the Chinese-lab models in this list
# for free. It requires an Alibaba Cloud account with real-name verification, which can be
# a barrier depending on your region/documents; that's the trade-off for $0 cost.
# Swap any entry back to "backend": "openrouter" (see the commented paid slug in each entry)
# if you'd rather pay for higher throughput / no verification hassle / no shared daily cap.
MODELSCOPE_BASE_URL = "https://api-inference.modelscope.cn/v1"

MODEL_REGISTRY = [
    {
        "key": "qwen3-vl-235b-a22b-instruct",
        "label": "Qwen3-VL 235B-A22B (Instruct) — via ModelScope, free tier",
        "backend": "openai_compatible",
        "model_id": "Qwen/Qwen3-VL-235B-A22B-Instruct",
        "base_url": MODELSCOPE_BASE_URL,
        "api_key_env": "MODELSCOPE_API_KEY",
        # Paid alternative: backend="openrouter", model_id="qwen/qwen3-vl-235b-a22b-instruct", $0.20/$0.88 per 1M
        "price_per_1m_usd": {"input": 0.0, "output": 0.0},
        "verified": True,
        "enabled": True,
    },
    {
        "key": "qwen3-vl-235b-a22b-thinking",
        "label": "Qwen3-VL 235B-A22B (Thinking)",
        "backend": "openrouter",  # not confirmed on ModelScope's free list; using paid OpenRouter
        "model_id": "qwen/qwen3-vl-235b-a22b-thinking",
        "price_per_1m_usd": {"input": 0.20, "output": 0.88},  # not free; thinking output tokens add up fast
        "verified": True,
        "enabled": True,
    },
    {
        "key": "qwen3-vl-8b-instruct",
        "label": "Qwen3-VL 8B (Instruct) — via ModelScope, free tier; cheap smoke-test model",
        "backend": "openai_compatible",
        "model_id": "Qwen/Qwen3-VL-8B-Instruct",
        "base_url": MODELSCOPE_BASE_URL,
        "api_key_env": "MODELSCOPE_API_KEY",
        # Paid alternative: backend="openrouter", model_id="qwen/qwen3-vl-8b-instruct", $0.117/$0.455 per 1M
        "price_per_1m_usd": {"input": 0.0, "output": 0.0},
        "verified": True,
        "enabled": True,
    },
    {
        # InternVL3.5 is NOT listed on OpenRouter at all (only older InternVL3 2B/14B/78B
        # are, under the opengvlab namespace) -- ModelScope's free tier is the only easy
        # hosted route for this one.
        "key": "internvl3.5-241b-a28b",
        "label": "InternVL3.5 241B-A28B — via ModelScope, free tier",
        "backend": "openai_compatible",
        "model_id": "OpenGVLab/InternVL3_5-241B-A28B",
        "base_url": MODELSCOPE_BASE_URL,
        "api_key_env": "MODELSCOPE_API_KEY",
        "price_per_1m_usd": {"input": 0.0, "output": 0.0},
        "verified": True,
        "enabled": True,
    },
    {
        "key": "glm-4.6v",
        "label": "GLM-4.6V 106B-A12B",
        "backend": "openrouter",
        # ZhipuAI publishes several GLM chat models free on ModelScope (GLM-4.7-Flash,
        # GLM-5.x); GLM-4.6V specifically wasn't confirmed there -- run
        # `GET https://api-inference.modelscope.cn/v1/models` yourself to check before
        # assuming this needs the paid OpenRouter route below.
        "model_id": "z-ai/glm-4.6v",
        "price_per_1m_usd": {"input": 0.30, "output": 0.90},  # not free
        "verified": True,
        "enabled": True,
    },
    {
        "key": "pixtral-large",
        "label": "Pixtral Large",
        "backend": "openrouter",
        # No free route found anywhere for this one -- Mistral doesn't publish to
        # ModelScope, and it's not on OpenRouter's free tier. Priciest model here; consider
        # skipping it in early sweeps.
        "model_id": "mistralai/pixtral-large-2411",
        "price_per_1m_usd": {"input": 2.00, "output": 6.00},
        "verified": True,
        "enabled": True,
    },
    {
        # Two independent free routes exist for Maverick: OpenRouter's ":free" slug
        # (used here, no account-verification hassle but shares OpenRouter's global free
        # quota and is tightly rate-limited) or ModelScope's
        # "LLM-Research/Llama-4-Maverick-17B-128E-Instruct" (higher daily cap, needs
        # real-name verification). Pick whichever friction you'd rather deal with.
        "key": "llama-4-maverick",
        "label": "Llama 4 Maverick — OpenRouter free slug",
        "backend": "openrouter",
        "model_id": "meta-llama/llama-4-maverick:free",  # paid alternative: "meta-llama/llama-4-maverick" ($0.20/$0.696 per 1M)
        "price_per_1m_usd": {"input": 0.0, "output": 0.0},
        "verified": True,
        "enabled": True,
    },
    {
        # Not on ModelScope's or OpenRouter's free catalog -- but genuinely small (16B
        # total / 3B active MoE), so self-hosting for free on a Colab/Kaggle GPU in 4-bit
        # is realistic, unlike the 100B+ models above. Start it with, e.g.:
        #   vllm serve moonshotai/Kimi-VL-A3B-Thinking-2506 --trust-remote-code \
        #       --served-model-name kimi-vl-thinking --limit-mm-per-prompt image=8
        # then point base_url at wherever that's running.
        "key": "kimi-vl-a3b-thinking",
        "label": "Kimi-VL-A3B-Thinking (self-host, free GPU-hours)",
        "backend": "openai_compatible",
        "model_id": "moonshotai/Kimi-VL-A3B-Thinking-2506",
        "base_url": "http://localhost:8000/v1",
        "api_key_env": "LOCAL_VLLM_API_KEY",
        "verified": False,
        "enabled": False,  # flip on once you're serving it locally
    },
    {
        # Not on ModelScope's free catalog (only DeepSeek's newer chat/reasoning lines are).
        # Trivially small to self-host, though: deepseek-vl2-tiny is 3.4B-MoE total with
        # only 1B activated -- runs comfortably on a free Colab/Kaggle T4.
        "key": "deepseek-vl2",
        "label": "DeepSeek-VL2 (self-host, free GPU-hours)",
        "backend": "openai_compatible",
        "model_id": "deepseek-ai/deepseek-vl2-tiny",  # or deepseek-vl2-small / deepseek-vl2 for more capacity
        "base_url": "http://localhost:8001/v1",
        "api_key_env": "LOCAL_VLLM_API_KEY",
        "verified": False,
        "enabled": False,
    },
    {
        # Western lab (Ai2) -- not on ModelScope. Self-host, or use the public HF Space
        # demo for small-scale interactive spot-checks (not suitable for a batch sweep).
        "key": "molmo2",
        "label": "Molmo2 (Ai2) (self-host, free GPU-hours)",
        "backend": "openai_compatible",
        "model_id": "allenai/Molmo2",
        "base_url": "http://localhost:8002/v1",
        "api_key_env": "LOCAL_VLLM_API_KEY",
        "verified": False,
        "enabled": False,
    },
]

print(f"{len(MODEL_REGISTRY)} models registered "
      f"({sum(m['enabled'] for m in MODEL_REGISTRY)} enabled by default).")

In [ ]:
def validate_openrouter_slugs(registry: list) -> None:
    """Best-effort check against OpenRouter's live catalog. Skips quietly if offline."""
    try:
        resp = requests.get("https://openrouter.ai/api/v1/models", timeout=20)
        resp.raise_for_status()
        known_ids = {m["id"] for m in resp.json().get("data", [])}
    except Exception as exc:
        print(f"Could not reach OpenRouter to validate slugs ({exc}); skipping.")
        return
    for m in registry:
        if m["backend"] != "openrouter":
            continue
        status = "OK" if m["model_id"] in known_ids else "NOT FOUND -- update model_id"
        print(f"{m['key']:32s} {m['model_id']:42s} {status}")


validate_openrouter_slugs(MODEL_REGISTRY)

## Step 4 — Unified vision-chat client

One thin OpenAI-compatible client covers both backends: only the base URL, auth header, and
model slug change between models. Retries with exponential backoff on rate limits/timeouts.

In [ ]:
class VisionChatClient:
    def __init__(self, model_cfg: dict, timeout: int = REQUEST_TIMEOUT_S, max_retries: int = MAX_RETRIES):
        self.cfg = model_cfg
        self.timeout = timeout
        self.max_retries = max_retries

        if model_cfg["backend"] == "openrouter":
            self.url = "https://openrouter.ai/api/v1/chat/completions"
            api_key = os.environ.get("OPENROUTER_API_KEY")
            if not api_key:
                raise RuntimeError("Set OPENROUTER_API_KEY (env var or .env) before querying OpenRouter models.")
            self.headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
        elif model_cfg["backend"] == "openai_compatible":
            self.url = model_cfg["base_url"].rstrip("/") + "/chat/completions"
            api_key = os.environ.get(model_cfg.get("api_key_env", ""), "EMPTY")
            self.headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
        else:
            raise ValueError(f"Unknown backend: {model_cfg['backend']}")

    @staticmethod
    def encode_image(path: Path) -> str:
        with open(path, "rb") as f:
            return base64.b64encode(f.read()).decode("utf-8")

    def query(self, image_path: Path, prompt: str, max_tokens: int = 512) -> str:
        image_b64 = self.encode_image(image_path)
        payload = {
            "model": self.cfg["model_id"],
            "temperature": 0,
            "max_tokens": max_tokens,
            "messages": [
                {
                    "role": "system",
                    "content": (
                        "You are answering a visual-reasoning benchmark question about the attached "
                        "image. Follow the question's requested answer format exactly. After any brief "
                        "reasoning, end your response with a final line in the exact form: "
                        "FINAL ANSWER: <answer>"
                    ),
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
                    ],
                },
            ],
        }
        last_exc = None
        for attempt in range(self.max_retries):
            try:
                resp = requests.post(self.url, headers=self.headers, json=payload, timeout=self.timeout)
                if resp.status_code == 429:
                    time.sleep(2 ** attempt)
                    continue
                resp.raise_for_status()
                return resp.json()["choices"][0]["message"]["content"]
            except Exception as exc:
                last_exc = exc
                time.sleep(min(30, 2 ** attempt))
        raise RuntimeError(f"Query failed after {self.max_retries} attempts: {last_exc}")

## Step 5 — Load closed L1–L5 questions for a domain

Joins the **public** `question_set.csv` (`question_id, task, image, prompt` — no answers) with
the **private** `answer_key.csv` (adds `groundtruth`) purely for local scoring; only the public
prompt text and image are ever sent to a model. Rows whose image is missing or still an
un-pulled LFS pointer are dropped with a warning rather than silently skipped.

In [ ]:
def load_domain_closed_questions(domain: str, sample_per_level, seed: int = RANDOM_SEED) -> pd.DataFrame:
    paths = ALL_DOMAINS[domain]
    questions = pd.read_csv(paths["question_set"])   # public: question_id, task, image, prompt
    answers = pd.read_csv(paths["answer_key"])         # private: + groundtruth

    df = questions.merge(answers[["question_id", "groundtruth"]], on="question_id", how="left")
    df["level"] = df["question_id"].str.extract(r"_q(\d)$").astype(int)
    df["domain"] = domain
    df["image_path"] = df["image"].apply(lambda name: paths["images_dir"] / name)

    def image_ready(p: Path) -> bool:
        return p.is_file() and not is_lfs_pointer(p)

    before = len(df)
    df = df[df["image_path"].apply(image_ready)].copy()
    if len(df) < before:
        print(f"[{domain}] dropped {before - len(df)}/{before} rows: image missing or not pulled from LFS")

    if sample_per_level is not None:
        df = (
            df.groupby("level", group_keys=False)
            .apply(lambda g: g.sample(n=min(sample_per_level, len(g)), random_state=seed))
        )
    return df.reset_index(drop=True)

## Step 6 — Recover per-question answer format / tolerance (for scoring only)

`question_set.csv`/`answer_key.csv` don't carry `answer_format` (only the combined Parquet/CSV
files do, and those are LFS pointers here too). Per-domain `annotations.jsonl` has it inline per
question, so we parse that file directly — never anything the model receives — to recover
numeric tolerances (e.g. angle_estimation's ±5°). Some `annotations.jsonl` files are themselves
LFS-tracked (see `.gitattributes`); for those this returns `{}` and scoring falls back to a
generic comparator.

In [ ]:
def load_answer_formats(domain: str) -> dict:
    path = ALL_DOMAINS[domain]["annotations"]
    if not path.is_file() or is_lfs_pointer(path):
        return {}
    formats = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue
            for q in row.get("questions", []):
                if "question_id" in q and "answer_format" in q:
                    formats[q["question_id"]] = q["answer_format"]
    return formats

## Step 7 — Scoring

A generic comparator that:

- normalizes whitespace/case/brackets on both sides,
- applies the stored `absolute_tolerance` when both sides parse as numbers (falling back to a
  5% relative tolerance if no tolerance was recovered in Step 6),
- does exact component-wise matching for comma-separated multi-part answers
  (e.g. `"190,no"` from `angle_estimation`'s L5 questions), and
- otherwise falls back to normalized exact match / substring match.

This is intentionally simple and will misgrade some free-form phrasing — it's meant as a fast
first pass across 34 heterogeneous domains, not a replacement for spot-checking `raw_response`.

In [ ]:
def normalize_text(s) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[.\s]+$", "", s)
    s = re.sub(r"^[{(\[]|[)}\]]$", "", s)
    return s.strip()


def try_float(s) -> float | None:
    try:
        return float(re.sub(r"[^0-9eE+\-.]", "", str(s)))
    except (ValueError, TypeError):
        return None


def score_answer(prediction: str, groundtruth, answer_format=None) -> dict:
    pred_norm = normalize_text(prediction)
    gt_norm = normalize_text(groundtruth)

    tolerance = None
    if isinstance(answer_format, dict) and answer_format.get("type") == "numeric_tolerance":
        tolerance = answer_format.get("absolute_tolerance")

    pred_val, gt_val = try_float(pred_norm), try_float(gt_norm)
    if pred_val is not None and gt_val is not None:
        tol = tolerance if tolerance is not None else max(0.5, abs(gt_val) * 0.05)
        return {"correct": abs(pred_val - gt_val) <= tol, "mode": "numeric", "tolerance_used": tol}

    if "," in gt_norm and "," in pred_norm:
        gt_parts = [p.strip() for p in gt_norm.split(",")]
        pred_parts = [p.strip() for p in pred_norm.split(",")]
        return {"correct": gt_parts == pred_parts, "mode": "multi_part_exact"}

    correct = pred_norm == gt_norm or (gt_norm != "" and gt_norm in pred_norm)
    return {"correct": correct, "mode": "text_exact_or_substring"}

## Step 8 — Extract the model's final answer from its raw response

The system prompt asks every model to end with `FINAL ANSWER: <answer>`. Not every model will
follow this reliably, so we fall back to the last non-empty line of the response.

In [ ]:
def extract_final_answer(raw_text: str) -> str:
    match = re.search(r"final answer\s*[:\-]\s*(.+)", raw_text, re.IGNORECASE)
    if match:
        return match.group(1).strip().splitlines()[0]
    lines = [l for l in raw_text.strip().splitlines() if l.strip()]
    return lines[-1] if lines else ""

## Step 9 — Runner: query, score, and cache

Results are appended to `eval_results/<model_key>/<domain>.jsonl`. Rerunning is resumable:
already-answered `question_id`s are skipped, so interrupting a long sweep and restarting the
cell later just picks up where it left off.

In [ ]:
def load_cached_results(model_key: str, domain: str, suffix: str = "") -> pd.DataFrame:
    path = RESULTS_ROOT / model_key / f"{domain}{suffix}.jsonl"
    if not path.is_file():
        return pd.DataFrame()
    return pd.read_json(path, lines=True)


def run_model_on_domain(model_cfg: dict, domain: str, df: pd.DataFrame, resume: bool = True) -> pd.DataFrame:
    out_dir = RESULTS_ROOT / model_cfg["key"]
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{domain}.jsonl"

    done_ids = set()
    if resume and out_path.is_file():
        with open(out_path, encoding="utf-8") as f:
            for line in f:
                try:
                    done_ids.add(json.loads(line)["question_id"])
                except (json.JSONDecodeError, KeyError):
                    continue

    todo = df[~df["question_id"].isin(done_ids)]
    if todo.empty:
        print(f"[{model_cfg['key']}/{domain}] all {len(df)} cached, skipping")
        return load_cached_results(model_cfg["key"], domain)

    client = VisionChatClient(model_cfg)
    answer_formats = load_answer_formats(domain)

    def process_row(row):
        try:
            raw = client.query(row["image_path"], row["prompt"])
        except Exception as exc:
            return {
                "question_id": row["question_id"], "domain": domain, "level": int(row["level"]),
                "model": model_cfg["key"], "raw_response": None, "prediction": None,
                "groundtruth": row["groundtruth"], "correct": None, "score_mode": None,
                "error": str(exc),
            }
        prediction = extract_final_answer(raw)
        result = score_answer(prediction, row["groundtruth"], answer_formats.get(row["question_id"]))
        return {
            "question_id": row["question_id"],
            "domain": domain,
            "level": int(row["level"]),
            "model": model_cfg["key"],
            "raw_response": raw,
            "prediction": prediction,
            "groundtruth": row["groundtruth"],
            "correct": result["correct"],
            "score_mode": result["mode"],
            "error": None,
        }

    with open(out_path, "a", encoding="utf-8") as f, ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = [pool.submit(process_row, row) for _, row in todo.iterrows()]
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f"{model_cfg['key']}/{domain}"):
            record = fut.result()
            f.write(json.dumps(record) + "\n")
            f.flush()

    return load_cached_results(model_cfg["key"], domain)

## Step 10 — Orchestration: smoke test, then the full sweep

With `SMOKE_TEST = True` (the default), this runs only `SMOKE_TEST_MODELS` x `SMOKE_TEST_DOMAINS`
at `SMOKE_TEST_SAMPLE_PER_LEVEL` images/level — enough to confirm API keys, image loading, and
scoring all work before spending real money. Once satisfied, set `SMOKE_TEST = False` in Step 0's
config cell (raise `SAMPLE_PER_LEVEL` gradually, and flip on `enabled: True` for whichever models
in `MODEL_REGISTRY` you've set up) and re-run this cell.

In [ ]:
if SMOKE_TEST:
    active_models = [m for m in MODEL_REGISTRY if m["key"] in SMOKE_TEST_MODELS]
    active_domains = SMOKE_TEST_DOMAINS
    sample_per_level = SMOKE_TEST_SAMPLE_PER_LEVEL
else:
    active_models = [m for m in MODEL_REGISTRY if m["enabled"]]
    active_domains = list(ALL_DOMAINS.keys())
    sample_per_level = SAMPLE_PER_LEVEL

print(f"Running {len(active_models)} model(s) x {len(active_domains)} domain(s), "
      f"{sample_per_level} images/level ({sample_per_level * 5} questions/domain/model).")

for model_cfg in active_models:
    for domain in active_domains:
        question_df = load_domain_closed_questions(domain, sample_per_level)
        if question_df.empty:
            print(f"Skipping {domain}: no images available locally (check Git LFS pull).")
            continue
        run_model_on_domain(model_cfg, domain, question_df)

## Step 11 — Aggregate results and compare against constant-answer baselines

Per the repo's own guidance, raw accuracy can be misleading on domains with imbalanced answer
distributions (e.g. `optical_illusion` L1, `gear_train` L2). The baseline here is computed as the
majority-class frequency in each domain/level's **full** local `answer_key.csv` — not just the
sampled subset — so it matches what you'd see at full scale even when running on a small sample.

In [ ]:
def load_all_results(suffix: str = "") -> pd.DataFrame:
    frames = []
    if not RESULTS_ROOT.exists():
        return pd.DataFrame()
    for model_dir in RESULTS_ROOT.iterdir():
        if not model_dir.is_dir():
            continue
        pattern = f"*{suffix}.jsonl"
        for jsonl_path in model_dir.glob(pattern):
            df = pd.read_json(jsonl_path, lines=True)
            if not df.empty:
                frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def domain_level_baseline(domain: str) -> dict:
    answers = pd.read_csv(ALL_DOMAINS[domain]["answer_key"])
    answers["level"] = answers["question_id"].str.extract(r"_q(\d)$").astype(int)
    baseline = {}
    for level, group in answers.groupby("level"):
        baseline[level] = group["groundtruth"].astype(str).value_counts(normalize=True).iloc[0]
    return baseline


results_df = load_all_results()
n_models = results_df["model"].nunique() if not results_df.empty else 0
n_domains = results_df["domain"].nunique() if not results_df.empty else 0
print(f"Loaded {len(results_df)} scored responses across {n_models} model(s) and {n_domains} domain(s).")

if not results_df.empty:
    scored = results_df[results_df["correct"].notna()]
    accuracy_by_model_domain = scored.groupby(["model", "domain"])["correct"].mean().unstack("domain")
    accuracy_by_model_level = scored.groupby(["model", "level"])["correct"].mean().unstack("level")
    display(accuracy_by_model_domain)
    display(accuracy_by_model_level)

In [ ]:
if not results_df.empty:
    baseline_rows = []
    for domain in results_df["domain"].unique():
        for level, baseline_acc in domain_level_baseline(domain).items():
            baseline_rows.append({"domain": domain, "level": level, "baseline_accuracy": baseline_acc})
    baseline_df = pd.DataFrame(baseline_rows)

    comparison = (
        scored.groupby(["model", "domain", "level"])["correct"].mean()
        .reset_index()
        .merge(baseline_df, on=["domain", "level"])
    )
    comparison["above_baseline"] = comparison["correct"] - comparison["baseline_accuracy"]
    display(comparison.sort_values("above_baseline"))

## Step 12 — Visualize

In [ ]:
import matplotlib.pyplot as plt

if not results_df.empty:
    fig, ax = plt.subplots(
        figsize=(max(8, len(accuracy_by_model_domain.columns) * 0.6), max(4, len(accuracy_by_model_domain) * 0.6))
    )
    im = ax.imshow(accuracy_by_model_domain.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
    ax.set_xticks(range(len(accuracy_by_model_domain.columns)))
    ax.set_xticklabels(accuracy_by_model_domain.columns, rotation=90)
    ax.set_yticks(range(len(accuracy_by_model_domain.index)))
    ax.set_yticklabels(accuracy_by_model_domain.index)
    ax.set_title("Accuracy by model x domain")
    fig.colorbar(im, ax=ax, label="accuracy")
    plt.tight_layout()
    plt.show()

    accuracy_by_model_level.T.plot(marker="o", figsize=(8, 5))
    plt.xlabel("Difficulty level (L1-L5)")
    plt.ylabel("Accuracy")
    plt.title("Accuracy by difficulty level")
    plt.legend(title="model", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

In [ ]:
summary_path = RESULTS_ROOT / "summary.json"
if not results_df.empty:
    summary = {
        "generated_from_rows": len(results_df),
        "accuracy_by_model_domain": accuracy_by_model_domain.round(4).to_dict(),
        "accuracy_by_model_level": accuracy_by_model_level.round(4).to_dict(),
    }
    summary_path.write_text(json.dumps(summary, indent=2))
    print(f"Saved summary to {summary_path}")

## Optional — Open-ended track (1 question/image, 100,000 total)

Reuses the same `VisionChatClient` against `open_questions.csv` (public prompt) and scores
against `open_answer_key.csv`'s stored sub-facts and tolerances. Per `OPEN_QUESTION_SPEC.md`
these are free-form justify-then-score prompts, so this scorer only checks whether each stored
sub-fact value shows up in the response (within tolerance for numeric fields) — it is **not**
a substitute for the human/LLM-judge grading a real open-ended evaluation needs, especially for
the justification and confidence-score portions of each answer.

In [ ]:
def load_domain_open_questions(domain: str, sample_n, seed: int = RANDOM_SEED) -> pd.DataFrame:
    paths = ALL_DOMAINS[domain]
    if not paths["open_questions"].is_file():
        return pd.DataFrame()

    questions = pd.read_csv(paths["open_questions"])   # public: question_id, image, prompt
    answers = pd.read_csv(paths["open_answer_key"])     # private: + acceptance_set/tolerances/targets/subfacts
    df = questions.merge(answers, on=["question_id", "image"], how="left", suffixes=("", "_ans"))
    df["domain"] = domain
    df["image_path"] = df["image"].apply(lambda name: paths["images_dir"] / name)
    df = df[df["image_path"].apply(lambda p: p.is_file() and not is_lfs_pointer(p))].copy()

    if sample_n is not None and len(df) > sample_n:
        df = df.sample(n=sample_n, random_state=seed)
    return df.reset_index(drop=True)


def score_open_answer(raw_response: str, row: pd.Series) -> dict:
    tolerances = {}
    try:
        tolerances = json.loads(row.get("tolerances", "{}") or "{}")
    except (json.JSONDecodeError, TypeError):
        pass

    meta_cols = {"question_id", "image", "acceptance_set", "tolerances", "targets", "prompt"}
    subfact_cols = [c for c in row.index if c not in meta_cols and pd.notna(row[c]) and row[c] != ""]

    numbers_in_response = [try_float(tok) for tok in re.findall(r"-?\d+\.?\d*", raw_response or "")]
    hits, total = 0, 0
    for col in subfact_cols:
        gt_value = row[col]
        total += 1
        gt_val_f = try_float(gt_value)
        if gt_val_f is not None:
            tol = (tolerances.get(col, {}) or {}).get("absolute_tolerance", max(0.5, abs(gt_val_f) * 0.05))
            if any(n is not None and abs(n - gt_val_f) <= tol for n in numbers_in_response):
                hits += 1
        elif normalize_text(gt_value) in normalize_text(raw_response or ""):
            hits += 1

    return {"subfacts_matched": hits, "subfacts_total": total,
            "subfact_match_rate": hits / total if total else None}


def run_model_on_domain_open(model_cfg: dict, domain: str, df: pd.DataFrame, resume: bool = True) -> pd.DataFrame:
    out_dir = RESULTS_ROOT / model_cfg["key"]
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{domain}_open.jsonl"

    done_ids = set()
    if resume and out_path.is_file():
        with open(out_path, encoding="utf-8") as f:
            for line in f:
                try:
                    done_ids.add(json.loads(line)["question_id"])
                except (json.JSONDecodeError, KeyError):
                    continue

    todo = df[~df["question_id"].isin(done_ids)]
    if todo.empty:
        print(f"[{model_cfg['key']}/{domain} open] all {len(df)} cached, skipping")
        return load_cached_results(model_cfg["key"], domain, suffix="_open")

    client = VisionChatClient(model_cfg)

    def process_row(row):
        try:
            raw = client.query(row["image_path"], row["prompt"], max_tokens=400)
        except Exception as exc:
            return {"question_id": row["question_id"], "domain": domain, "model": model_cfg["key"],
                     "raw_response": None, "error": str(exc)}
        scored = score_open_answer(raw, row)
        return {
            "question_id": row["question_id"],
            "domain": domain,
            "model": model_cfg["key"],
            "raw_response": raw,
            **scored,
            "error": None,
        }

    with open(out_path, "a", encoding="utf-8") as f, ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = [pool.submit(process_row, row) for _, row in todo.iterrows()]
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f"{model_cfg['key']}/{domain} open"):
            record = fut.result()
            f.write(json.dumps(record) + "\n")
            f.flush()

    return load_cached_results(model_cfg["key"], domain, suffix="_open")


# Uncomment to run the open-ended track over the same active_models / active_domains from Step 10:
# OPEN_SAMPLE_N = SMOKE_TEST_SAMPLE_PER_LEVEL if SMOKE_TEST else SAMPLE_PER_LEVEL
# for model_cfg in active_models:
#     for domain in active_domains:
#         open_df = load_domain_open_questions(domain, OPEN_SAMPLE_N)
#         if open_df.empty:
#             continue
#         run_model_on_domain_open(model_cfg, domain, open_df)

## Notes, limitations, and how to scale up

**Before a real run**
- `git lfs pull` so `images/*.png` are real bytes (Step 1 will keep warning otherwise).
- Re-run the OpenRouter slug validator (Step 3) — model catalogs move fast and slugs marked
  `verified: False` (`internvl3.5-241b-a28b`, `glm-4.6v`) may need a one-line update.
- For the 3 models shipped `enabled: False` (`kimi-vl-a3b-thinking`, `deepseek-vl2`, `molmo2`),
  stand up a local OpenAI-compatible server (e.g. `vllm serve <hf-repo> --port 8000`), point
  `base_url` at it, and flip `enabled: True`.

**Scaling from smoke test to full suite**
1. `SMOKE_TEST = True` on 1 model, 1 domain, 2 images/level — confirms auth, image loading, and
   scoring end-to-end.
2. Raise `SAMPLE_PER_LEVEL` to ~20–50 with `SMOKE_TEST = False` but only 2–3 cheap models
   enabled — get a first read on per-domain difficulty and cost per model before committing.
3. Enable all 10 models and raise `SAMPLE_PER_LEVEL` toward 3000 (the full domain size) only once
   cost and latency are predictable. The full suite is 500,000 questions x 10 models; budget
   accordingly and prefer running domains in separate batches over multiple sessions — the
   caching in Step 9 makes this safe to pause and resume.

**Known limitations of this harness**
- `score_answer` and `score_open_answer` are heuristic. Multi-part, letter-list
  (`letter_any_of_list`), and free-text justification answers will have some false negatives/positives
  — always spot-check `raw_response` for a sample of "incorrect" answers before trusting a number.
- Never point any of these clients at `annotations.jsonl`, `answer_key.csv`, `open_answer_key.csv`,
  `open_annotations.jsonl`, or the combined answer files as model input — they're read locally for
  scoring only, exactly as the repo's own evaluation guidance requires.
- `projectile_motion_dataset_1000` has only 1,000 images (5,000 questions); everywhere else
  assumes 3,000 images, but `load_domain_closed_questions` doesn't hardcode this, so it's handled
  automatically.